# Private Domain: Using `routingDomain` with AgentCore Gateway

This lab demonstrates how to connect [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) to a resource that uses a **private domain**, a domain that only resolves inside the VPC via a [Route 53 private hosted zone](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/hosted-zones-private.html).

## Architecture (this lab)

![arch](./images/private-domain.png)

The target URL uses the domain covered by the ALB's public certificate. VPC Lattice routes traffic via the ALB's DNS (`routingDomain`). No public DNS record is needed for the target domain.

For background on VPC egress and managed VPC Lattice, see the [project README](../README.md) and [Advanced Concepts README](./README.md).

## The Problem

Amazon VPC Lattice requires a **publicly resolvable domain** for its resource configuration. If your resource uses a domain that only resolves inside the VPC (private hosted zone), VPC Lattice cannot create the resource configuration.

## The Solution: `routingDomain`

The `routingDomain` field provides a publicly resolvable domain that VPC Lattice uses for its resource configuration and routing. At invocation time, VPC Lattice routes traffic through the `routingDomain` while setting the TLS SNI to the actual target domain, so the TLS handshake succeeds against your resource's certificate.

### When your private domain already points to an ALB, NLB, or VPCE

If your private hosted zone resolves to a load balancer or VPC endpoint, you already have a publicly resolvable DNS name you can use as `routingDomain`:

| Resource | routingDomain |
|----------|---------------|
| **Internal ALB** | `internal-<name>-<id>.<region>.elb.amazonaws.com` |
| **Internal NLB** | `internal-<name>-<id>.elb.<region>.amazonaws.com` |
| **VPC Endpoint** | `<vpce-id>.execute-api.<region>.vpce.amazonaws.com` |

These DNS names are publicly resolvable (they resolve to private IPs) and satisfy VPC Lattice's requirement. **This is the easy case**, no additional infrastructure needed.

### When your private domain points directly to an IP address

If your private hosted zone resolves to an EC2 private IP (or any resource without a publicly resolvable DNS name), there is no intermediate to use as `routingDomain`. The workaround is to **place an internal ALB in front** of the resource:

1. Deploy an internal ALB with a public ACM certificate
2. Point the ALB at your backend resource
3. Use the ALB's DNS name as `routingDomain`

This is covered in the [private certificate authority](./02-private-certificate-authority.ipynb) and [self-signed certificate](./03-self-signed-certificate.ipynb) labs.

## Prerequisites

- Completed [Lab 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) (VPC + AgentCore Gateway deployed)
- An [ACM public certificate](../00-prerequisites/create-acm-public-certificate.md) for a domain you own

## Step 1: Install dependencies and import libraries

In [ ]:
import os
from pathlib import Path

# Change to project root
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# Restore variables from Lab 0
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Get Cognito client secret
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(
    UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID
)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## Step 2: Configure domain and certificate

Provide your ACM public certificate ARN and the domain it covers. The domain must match the certificate, for example, if your cert is for `*.internal.yourcompany.com`, enter a subdomain like `api.internal.yourcompany.com`.

This domain is used as the AgentCore Gateway target URL. You do **not** need a public DNS record for it, `routingDomain` handles routing via the ALB DNS.

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ")
DOMAIN = input(
    "Domain name covered by the certificate (e.g., api.internal.yourcompany.com): "
)

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## Step 3: Deploy infrastructure

This stack deploys:
- An **EC2 instance** running a REST API (FastAPI) on HTTP port 8000
- An **internal ALB** with your public ACM certificate on HTTPS port 443, forwarding HTTP to the EC2
- A **Route 53 private hosted zone** (`internal.{baseDomain}`) with:
  - `api.internal.{baseDomain}`, Alias record pointing to the ALB (Scenario 1)
  - `direct.internal.{baseDomain}`, A record pointing to the EC2 private IP (Scenario 2)

Inside the VPC, `api.internal.{baseDomain}` resolves to the ALB. Outside the VPC, this domain does not resolve, hence the need for `routingDomain`.

In [ ]:
!cdk deploy PrivateDomain -c publicCertArn={CERT_ARN} --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file pd-outputs.json

In [ ]:
with open("pd-outputs.json") as f:
    pd_outputs = json.load(f)["PrivateDomain"]

ALB_DNS = pd_outputs["AlbDnsName"]
ALB_SG_ID = pd_outputs["AlbSgId"]
API_KEY_VALUE = pd_outputs["ApiKey"]
EC2_INSTANCE_ID = pd_outputs["Ec2InstanceId"]
EC2_PRIVATE_IP = pd_outputs["Ec2PrivateIp"]
PRIVATE_DOMAIN_ALB = pd_outputs["PrivateDomainAlb"]
PRIVATE_DOMAIN_DIRECT = pd_outputs["PrivateDomainDirect"]

print(f"ALB DNS:                {ALB_DNS}")
print(f"ALB SG:                 {ALB_SG_ID}")
print(f"EC2 instance:           {EC2_INSTANCE_ID}")
print(f"EC2 private IP:         {EC2_PRIVATE_IP}")
print(f"Private domain (ALB):   {PRIVATE_DOMAIN_ALB}")
print(f"Private domain (EC2):   {PRIVATE_DOMAIN_DIRECT}")

## Scenario 1: Private domain resolves to ALB / NLB / VPCE

Our private domain `api.internal.{baseDomain}` resolves to the ALB inside the VPC. The ALB has a public certificate. The ALB's DNS name (`internal-*.elb.amazonaws.com`) is **publicly resolvable**, we use it as `routingDomain`.

```
Inside VPC:  api.internal.{baseDomain} to ALB private IPs
Outside VPC: api.internal.{baseDomain} to does not resolve
Anywhere:    internal-*.elb.amazonaws.com to ALB private IPs  (publicly resolvable)
```

This pattern works the same way for **NLBs** (`internal-*.elb.*.amazonaws.com`) and **VPC endpoints** (`*.vpce.amazonaws.com`), any privately-deployed resource that has a publicly resolvable DNS name can be used as `routingDomain`.

> **Key insight:** `routingDomain` is only used by VPC Lattice for DNS resolution and routing. The target URL domain (from your public cert) is what's sent as the TLS SNI. The two domains don't need to match.

## Step 4: Create AgentCore Gateway Target

- **Target URL** (`https://{DOMAIN}`), matches the public certificate on the ALB
- **`routingDomain`** (`{ALB_DNS}`), the publicly resolvable ALB DNS name

> **Security group:** We pass the ALB's security group to `securityGroupIds` so the Resource Gateway ENIs can reach the ALB on port 443.

In [ ]:
with open("03-advanced-concepts/openapi-private.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{DOMAIN}"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Server URL:     {TARGET_ENDPOINT}")
print(f"routingDomain:  {ALB_DNS}")
print(f"Endpoints:      {list(openapi_schema['paths'].keys())}")

In [ ]:
# Create API key credential provider
cred_response = agentcore.create_api_key_credential_provider(
    name="private-domain-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

In [ ]:
response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="private-domain",
    description="Private domain with public cert, routed via ALB routingDomain",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedLatticeResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [ALB_SG_ID],
            "routingDomain": ALB_DNS,
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(
        gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID
    )
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(
            f"  Managed resources: {target.get('privateEndpointManagedResources', {})}"
        )
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## Step 5: Invoke the API through AgentCore Gateway

Get an access token from Cognito, then invoke the API through the gateway. The traffic flows:

```
Your request to AgentCore Gateway to VPC Lattice (via ALB DNS) to ALB (public cert) to EC2 (:8000)
```

The private domain (`api.internal.{baseDomain}`) was never needed in the data path, only `routingDomain` (ALB DNS) was used for routing.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-domain___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Create an item
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "private-domain___createItem",
            "arguments": {"name": "Widget", "price": 9.99},
        },
        "id": 3,
    },
)
print("Create item:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-domain___listItems", "arguments": {}},
        "id": 4,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## Scenario 2: Private domain points directly to an IP address

Our stack also created `direct.internal.{baseDomain}` → EC2 private IP. This represents a common pattern where the private domain resolves directly to a compute resource, **no ALB, NLB, or VPCE** in front.

```
Inside VPC:  direct.internal.{baseDomain} to 10.0.x.x (EC2 private IP)
Outside VPC: direct.internal.{baseDomain} to does not resolve
```

In this case, there is **no publicly resolvable DNS name** to use as `routingDomain`. The workaround is to place an **internal ALB with a public certificate** in front of the resource:

1. Deploy an internal ALB with a public ACM certificate
2. Point the ALB at the backend (EC2 on port 8000)
3. Use the ALB DNS as `routingDomain`

This is exactly the pattern from Step 3, the ALB we already deployed serves as the publicly resolvable intermediate.

For a step-by-step walkthrough of adding an ALB in front of an existing resource, see:
- [Private Certificate Authority lab](./02-private-certificate-authority.ipynb), also covers private CA certs
- [Self-Signed Certificate lab](./03-self-signed-certificate.ipynb), also covers self-signed certs
- [Private Domain + Private Certificate lab](./04-private-domain-and-certificate.ipynb), covers both private DNS and private certs together

## Cleanup

1. Delete the gateway target
2. Destroy the CDK stack

In [ ]:
# # Step 1: Delete gateway target
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # Delete credential provider
# agentcore.delete_api_key_credential_provider(name="private-domain-api-key")
# print("Deleted credential provider: private-domain-api-key")

In [ ]:
# # Step 2: Destroy the stack
# !cdk destroy PrivateDomain -c publicCertArn={CERT_ARN} --profile {ACCOUNT_A_PROFILE} --force